# EDA Report — XuetangX Dataset

**Purpose:** Full exploratory data analysis of the XuetangX MOOC dataset, tracing every transformation from raw event log (NB01) through the meta-learning episode index (NB05).  
**Data source:** `./reports/*/report.json` — all numbers are read directly from pipeline run artefacts. No values are fabricated.

---

In [1]:
import json, os
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# ── Repo root resolution ──────────────────────────────────────────────
REPO_ROOT = Path.cwd()
for _ in range(10):
    if (REPO_ROOT / 'PROJECT_STATE.md').exists():
        break
    REPO_ROOT = REPO_ROOT.parent
REPORTS_DIR = REPO_ROOT / 'reports'
DATASET = 'xuetangx'

def load_latest(nb_name):
    d = REPORTS_DIR / nb_name
    if not d.exists():
        raise FileNotFoundError(f'No reports directory for {nb_name}')
    runs = sorted(d.iterdir())
    for run in reversed(runs):
        rp = run / 'report.json'
        if rp.exists():
            return json.loads(rp.read_text('utf-8'))
    raise FileNotFoundError(f'No report.json found under {d}')

print(f'REPO_ROOT  : {REPO_ROOT}')
print(f'REPORTS_DIR: {REPORTS_DIR}')
print(f'Dataset    : {DATASET.upper()}')

REPO_ROOT  : /home/user/jamalla/anonymous-users-mooc-session-meta
REPORTS_DIR: /home/user/jamalla/anonymous-users-mooc-session-meta/reports
Dataset    : XUETANGX


---
## Step 1 — Raw Data Ingestion (NB01)

The raw XuetangX log contains fine-grained learner interaction events spanning six months (Feb–Aug 2017).  
NB01 reads the source JSON, filters to learning events, and writes a cleaned Parquet file.

In [2]:
r01 = load_latest('01_ingest_xuetangx')
m01 = r01['metrics']

print('=' * 52)
print('NB01 — RAW INGESTION METRICS')
print('=' * 52)
print(f'  Total events ingested : {m01["n_events"]:>15,}')
print(f'  Unique users          : {m01["n_users"]:>15,}')
print(f'  Unique courses        : {m01["n_courses"]:>15,}')
print(f'  Event types           : {m01["n_event_types"]:>15,}')
print()
print('  Run timestamp :', r01['run_tag'])
print('  Created at    :', r01['created_at'])

# ── Event type distribution chart ─────────────────────────────────────
top_events = r01['sanity_samples']['top_event_types']
events_sorted = sorted(top_events.items(), key=lambda x: x[1], reverse=True)
labels  = [e[0].replace('_', ' ') for e in events_sorted]
counts  = [e[1] for e in events_sorted]
colors  = plt.cm.Blues(np.linspace(0.4, 0.85, len(labels)))

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(labels[::-1], counts[::-1], color=colors[::-1])
for bar, cnt in zip(bars, counts[::-1]):
    ax.text(bar.get_width() + 50000, bar.get_y() + bar.get_height()/2,
            f'{cnt:,}', va='center', fontsize=9)
ax.set_xlabel('Event count')
ax.set_title('XuetangX — Event Type Distribution (NB01)', fontsize=13, fontweight='bold')
ax.set_xlim(0, max(counts) * 1.15)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(REPO_ROOT / 'reports' / 'eda_xuetangx_01_event_types.png', dpi=150)
plt.show()
print('Chart saved: eda_xuetangx_01_event_types.png')

NB01 — RAW INGESTION METRICS
  Total events ingested :      28,002,537
  Unique users          :         182,755
  Unique courses        :           1,518
  Event types           :               8

  Run timestamp : 20260408_125223
  Created at    : 2026-04-08T12:52:23
Chart saved: eda_xuetangx_01_event_types.png


---
## Step 2 — Sessionization (NB02)

Events are grouped into sessions using a **30-minute inactivity threshold** (gap > 1800 s → new session).  
Sessions with fewer than 2 events are discarded. This step also assigns a `session_id` and computes session-level aggregates.

In [3]:
r02 = load_latest('02_sessionize_xuetangx')
m02 = r02['metrics']

print('=' * 52)
print('NB02 — SESSIONIZATION METRICS')
print('=' * 52)
print(f'  Events after filtering : {m02["n_events"]:>13,}')
print(f'  Sessions created       : {m02["n_sessions"]:>13,}')
print(f'  Users with sessions    : {m02["n_users"]:>13,}')
print(f'  Gap threshold          : {m02["gap_threshold_sec"]:>13,} s  (30 min)')
print(f'  Min events / session   : {m02["min_events_per_session"]:>13}')
print()
print('  Inter-event gap statistics:')
gs = m02['gap_stats']
print(f'    Total gaps measured  : {gs["n_gaps"]:>13,}')
print(f'    Median gap (p50)     : {gs["p50_sec"]:>13.1f} s')
print(f'    p90 gap              : {gs["p90_sec"]:>13.1f} s')
print(f'    p95 gap              : {gs["p95_sec"]:>13.1f} s')
print(f'    p99 gap              : {gs["p99_sec"]:>13.1f} s')
print(f'    Max gap              : {gs["max_sec"]:>13.1f} s')

# ── Gap percentile chart ───────────────────────────────────────────────
percentiles = ['p50\n(median)', 'p90', 'p95', 'p99', 'max']
values      = [gs['p50_sec'], gs['p90_sec'], gs['p95_sec'], gs['p99_sec'], gs['max_sec']]
colors_gap  = ['#4C9BE8', '#3A7EC8', '#2860A8', '#1A448A', '#0E2B6E']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: bar on linear scale (first 4 percentiles)
axes[0].bar(percentiles[:-1], values[:-1], color=colors_gap[:-1], edgecolor='white')
for i, v in enumerate(values[:-1]):
    axes[0].text(i, v + 80, f'{v:.0f}s', ha='center', fontsize=9)
axes[0].axhline(y=1800, color='red', linestyle='--', linewidth=1.2, label='Session gap threshold (1800 s)')
axes[0].set_ylabel('Gap duration (seconds)')
axes[0].set_title('Inter-event Gap Percentiles (p50–p99)', fontsize=11)
axes[0].legend(fontsize=8)
axes[0].spines[['top', 'right']].set_visible(False)

# Right: log scale to show full range including max
axes[1].bar(percentiles, values, color=colors_gap, edgecolor='white')
for i, v in enumerate(values):
    axes[1].text(i, v * 1.5, f'{v:,.0f}s', ha='center', fontsize=8)
axes[1].set_yscale('log')
axes[1].axhline(y=1800, color='red', linestyle='--', linewidth=1.2, label='Threshold')
axes[1].set_ylabel('Gap duration (log scale)')
axes[1].set_title('Gap Distribution — Log Scale (p50 to Max)', fontsize=11)
axes[1].legend(fontsize=8)
axes[1].spines[['top', 'right']].set_visible(False)

plt.suptitle('XuetangX — Session Gap Analysis (NB02)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(REPO_ROOT / 'reports' / 'eda_xuetangx_02_gap_stats.png', dpi=150)
plt.show()
print('Chart saved: eda_xuetangx_02_gap_stats.png')

NB02 — SESSIONIZATION METRICS
  Events after filtering :    27,875,155
  Sessions created       :       906,341
  Users with sessions    :       173,335
  Gap threshold          :         1,800 s  (30 min)
  Min events / session   :             2

  Inter-event gap statistics:
    Total gaps measured  :    27,533,251
    Median gap (p50)     :           4.0 s
    p90 gap              :         205.0 s
    p95 gap              :         504.0 s
    p99 gap              :       21022.0 s
    Max gap              :     7005723.0 s
Chart saved: eda_xuetangx_02_gap_stats.png


---
## Step 3 — Vocabulary & Sequence Pairs (NB03)

Each session is transformed into a sequence of course IDs visited in chronological order.  
For each position in the sequence, a *(prefix → next item)* pair is created — the target for next-item prediction.

In [4]:
r03 = load_latest('03_vocab_pairs_xuetangx')
m03 = r03['metrics']

print('=' * 52)
print('NB03 — VOCABULARY & PAIRS METRICS')
print('=' * 52)
print(f'  Vocabulary size (items)  : {m03["n_items"]:>12,}')
print(f'  Total (prefix, label) pairs : {m03["n_pairs"]:>9,}')
print(f'  Users with at least 1 pair  : {m03["n_users"]:>9,}')
print()
print('  Sample pairs (from sanity_samples):')
for p in r03['sanity_samples']['pairs_head3']:
    print(f'    user={p["user_id"]}  session={p["session_id"]}  '
          f'prefix_len={p["prefix_len"]}  label={p["label"]}')

# Compute avg pairs per user
avg_pairs = m03['n_pairs'] / m03['n_users']
avg_sessions = r02['metrics']['n_sessions'] / r02['metrics']['n_users']
print()
print(f'  Derived stats:')
print(f'    Avg pairs per user     : {avg_pairs:>10.1f}')
print(f'    Avg sessions per user  : {avg_sessions:>10.1f}')
print(f'    Pairs per session (avg): {m03["n_pairs"] / m02["n_sessions"]:>10.2f}')

# ── Users funnel from NB01 → NB03 ────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
stages = ['Raw users\n(NB01)', 'Users with\nsessions (NB02)', 'Users with\npairs (NB03)']
values = [m01['n_users'], m02['n_users'], m03['n_users']]
colors = ['#6baed6', '#3182bd', '#08519c']
bars = ax.bar(stages, values, color=colors, width=0.5, edgecolor='white')
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1500,
            f'{v:,}', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Number of users')
ax.set_title('XuetangX — User Count Funnel NB01 → NB03', fontsize=12, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(REPO_ROOT / 'reports' / 'eda_xuetangx_03_user_funnel.png', dpi=150)
plt.show()
print('Chart saved: eda_xuetangx_03_user_funnel.png')

NB03 — VOCABULARY & PAIRS METRICS
  Vocabulary size (items)  :        1,517
  Total (prefix, label) pairs :   487,696
  Users with at least 1 pair  :    70,809

  Sample pairs (from sanity_samples):
    user=1000063  session=1000063_000002  prefix_len=1  label=1284
    user=1000063  session=1000063_000002  prefix_len=2  label=939
    user=1000063  session=1000063_000002  prefix_len=3  label=1287

  Derived stats:
    Avg pairs per user     :        6.9
    Avg sessions per user  :        5.2
    Pairs per session (avg):       0.54
Chart saved: eda_xuetangx_03_user_funnel.png


---
## Step 4 — Session Reliability Scores / SRS (NB03b)

Each session receives an **SRS score ∈ [0, 1]** composed of three components:
- **Intensity**: `min(n_events / CAP_INTENSITY, 1.0)` — how actively the learner interacted
- **Extent**: `min(duration_sec / CAP_EXTENT, 1.0)` — how long the session lasted
- **Composition**: `n_unique_action_types / 6` — diversity of interaction types used

`SRS = (intensity + extent + composition) / 3`

CAPs are computed from training sessions only (no test leakage).  
High SRS → richer signal for meta-learning adaptation.

In [5]:
r03b = load_latest('03b_srs_scores_xuetangx')
m03b = r03b['metrics']

print('=' * 52)
print('NB03b — SRS SCORE METRICS')
print('=' * 52)
print(f'  Sessions scored        : {m03b["n_sessions"]:>14,}')
print(f'  Pairs with SRS         : {m03b["n_pairs"]:>14,}')
print(f'  CAP intensity (p95)    : {m03b["cap_intensity"]:>14.1f} events')
print(f'  CAP extent (p95)       : {m03b["cap_extent"]:>14.1f} sec')
print(f'  N_POSSIBLE action types: {6:>14}')
print()
print('  SRS distribution:')
print(f'    Mean                 : {m03b["srs_mean"]:>14.4f}')
print(f'    Median (p50)         : {m03b["srs_p50"]:>14.4f}')
print()
print('  Session tier breakdown:')
print(f'    Low  (SRS < 0.33)    : {m03b["tier_low"]*100:>13.1f}%')
print(f'    Med  (0.33-0.66)     : {m03b["tier_medium"]*100:>13.1f}%')
print(f'    High (SRS >= 0.66)   : {m03b["tier_high"]*100:>13.1f}%')

# ── SRS visualization ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Tier pie chart
tier_vals   = [m03b['tier_low'], m03b['tier_medium'], m03b['tier_high']]
tier_labels = [f'Low (<0.33)\n{m03b["tier_low"]*100:.1f}%',
               f'Medium (0.33-0.66)\n{m03b["tier_medium"]*100:.1f}%',
               f'High (>=0.66)\n{m03b["tier_high"]*100:.1f}%']
tier_colors = ['#d73027', '#fee090', '#4dac26']
wedges, _ = axes[0].pie(tier_vals, labels=tier_labels, colors=tier_colors,
                         startangle=140, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[0].set_title('SRS Tier Distribution', fontsize=11)

# Right: reconstructed SRS histogram from percentile stats
# Use known stats: mean=0.3248, p50=0.2456, span [0,1]
# Reconstruct approximate beta-like distribution using mean and median
r09 = load_latest('09_srs_validation_xuetangx')
m09 = r09['metrics']

percentile_labels = ['min', 'p25', 'p50', 'p75', 'max']
percentile_vals   = [m09['min'], m09['p25'], m09['p50'], m09['p75'], m09['max']]
axes[1].barh(percentile_labels, percentile_vals, color='#3182bd', edgecolor='white')
for i, v in enumerate(percentile_vals):
    axes[1].text(v + 0.01, i, f'{v:.4f}', va='center', fontsize=9)
axes[1].axvline(x=m09['mean'], color='red', linestyle='--', linewidth=1.5, label=f'Mean={m09["mean"]:.3f}')
axes[1].axvline(x=0.33, color='orange', linestyle=':', linewidth=1.2, label='Low/Med boundary (0.33)')
axes[1].axvline(x=0.66, color='green', linestyle=':', linewidth=1.2, label='Med/High boundary (0.66)')
axes[1].set_xlabel('SRS score')
axes[1].set_xlim(0, 1.08)
axes[1].set_title('SRS Score Percentiles (from NB09 Validation)', fontsize=11)
axes[1].legend(fontsize=8, loc='lower right')
axes[1].spines[['top', 'right']].set_visible(False)

plt.suptitle('XuetangX — Session Reliability Scores (NB03b + NB09)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(REPO_ROOT / 'reports' / 'eda_xuetangx_03b_srs.png', dpi=150)
plt.show()
print('Chart saved: eda_xuetangx_03b_srs.png')

NB03b — SRS SCORE METRICS
  Sessions scored        :        906,341
  Pairs with SRS         :        487,696
  CAP intensity (p95)    :          115.0 events
  CAP extent (p95)       :         6653.0 sec
  N_POSSIBLE action types:              6

  SRS distribution:
    Mean                 :         0.3248
    Median (p50)         :         0.2456

  Session tier breakdown:
    Low  (SRS < 0.33)    :          62.8%
    Med  (0.33-0.66)     :          25.8%
    High (SRS >= 0.66)   :          11.4%
Chart saved: eda_xuetangx_03b_srs.png


---
## Step 5 — User Split (NB04)

Users are split into disjoint **train / val / test** sets using a **70 / 15 / 15** ratio.  
This is a **cold-start** design: no user appears in more than one split. The model never sees test users during training.

In [6]:
r04 = load_latest('04_user_split_xuetangx')
m04 = r04['metrics']

print('=' * 60)
print('NB04 — USER SPLIT METRICS')
print('=' * 60)
print(f'  Total users            : {m04["n_users_total"]:>13,}')
print(f'  Train users (70%)      : {m04["n_users_train"]:>13,}  [{m04["n_users_train"]/m04["n_users_total"]*100:.1f}%]')
print(f'  Val users  (15%)       : {m04["n_users_val"]:>13,}  [{m04["n_users_val"]/m04["n_users_total"]*100:.1f}%]')
print(f'  Test users (15%)       : {m04["n_users_test"]:>13,}  [{m04["n_users_test"]/m04["n_users_total"]*100:.1f}%]')
print()
print(f'  Pairs — Train          : {m04["n_pairs_train"]:>13,}')
print(f'  Pairs — Val            : {m04["n_pairs_val"]:>13,}')
print(f'  Pairs — Test           : {m04["n_pairs_test"]:>13,}')
print(f'  Total pairs check      : {m04["n_pairs_train"]+m04["n_pairs_val"]+m04["n_pairs_test"]:>13,}')

# ── Split chart ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# User split
split_labels = ['Train (70%)', 'Val (15%)', 'Test (15%)']
user_counts  = [m04['n_users_train'], m04['n_users_val'], m04['n_users_test']]
pair_counts  = [m04['n_pairs_train'],  m04['n_pairs_val'],  m04['n_pairs_test']]
xpos = np.arange(3)
colors_split = ['#2166ac', '#74add1', '#abd9e9']

bars = axes[0].bar(xpos, user_counts, color=colors_split, edgecolor='white', width=0.5)
for bar, v in zip(bars, user_counts):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+300,
                 f'{v:,}', ha='center', fontsize=10, fontweight='bold')
axes[0].set_xticks(xpos)
axes[0].set_xticklabels(split_labels)
axes[0].set_ylabel('Users')
axes[0].set_title('Users per Split', fontsize=11)
axes[0].spines[['top', 'right']].set_visible(False)

bars = axes[1].bar(xpos, pair_counts, color=colors_split, edgecolor='white', width=0.5)
for bar, v in zip(bars, pair_counts):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+2000,
                 f'{v:,}', ha='center', fontsize=10, fontweight='bold')
axes[1].set_xticks(xpos)
axes[1].set_xticklabels(split_labels)
axes[1].set_ylabel('Pairs')
axes[1].set_title('Pairs per Split', fontsize=11)
axes[1].spines[['top', 'right']].set_visible(False)

plt.suptitle('XuetangX — User Split 70/15/15 (NB04)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPO_ROOT / 'reports' / 'eda_xuetangx_04_user_split.png', dpi=150)
plt.show()
print('Chart saved: eda_xuetangx_04_user_split.png')

NB04 — USER SPLIT METRICS
  Total users            :        70,809
  Train users (70%)      :        49,566  [70.0%]
  Val users  (15%)       :        10,621  [15.0%]
  Test users (15%)       :        10,622  [15.0%]

  Pairs — Train          :       344,532
  Pairs — Val            :        69,663
  Pairs — Test           :        73,501
  Total pairs check      :       487,696
Chart saved: eda_xuetangx_04_user_split.png


---
## Step 6 — Meta-Learning Episode Index (NB05)

For each user in each split, episodes are built using the **K=5 support + Q=10 query** protocol:  
- **Support set** (K=5): the earliest 5 pairs from the user — used for inner-loop adaptation  
- **Query set** (Q=10): the next 10 pairs — used for outer-loop loss / evaluation  
- A user needs at least **15 pairs** to form 1 episode. Only test users from NB04 contribute test episodes.

In [7]:
r05 = load_latest('05_episode_index_xuetangx')
m05 = r05['metrics']

print('=' * 52)
print('NB05 — EPISODE INDEX METRICS')
print('=' * 52)
print(f'  Episode protocol : K={m05["K"]} support, Q={m05["Q"]} query')
print(f'  Train episodes   : {m05["n_episodes_train"]:>12,}')
print(f'  Val episodes     : {m05["n_episodes_val"]:>12,}')
print(f'  Test episodes    : {m05["n_episodes_test"]:>12,}')
print()
print('  Derived stats:')
print(f'    Train episodes / train user : {m05["n_episodes_train"]/m04["n_users_train"]:>10.2f}')
print(f'    Val episodes / val user     : {m05["n_episodes_val"]/m04["n_users_val"]:>10.2f}')
print(f'    Test episodes / test user   : {m05["n_episodes_test"]/m04["n_users_test"]:>10.2f}')

# ── Episode bar chart ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
split_labels = ['Train', 'Val', 'Test']
episode_counts = [m05['n_episodes_train'], m05['n_episodes_val'], m05['n_episodes_test']]
colors_ep = ['#1b7837', '#a1d99b', '#31a354']
bars = ax.bar(split_labels, episode_counts, color=colors_ep, width=0.45, edgecolor='white')
for bar, v in zip(bars, episode_counts):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
            f'{v:,}', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Episode count')
ax.set_title(f'XuetangX — Episodes per Split (K={m05["K"]}, Q={m05["Q"]}) (NB05)',
             fontsize=12, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(REPO_ROOT / 'reports' / 'eda_xuetangx_05_episodes.png', dpi=150)
plt.show()
print('Chart saved: eda_xuetangx_05_episodes.png')

NB05 — EPISODE INDEX METRICS
  Episode protocol : K=5 support, Q=10 query
  Train episodes   :      113,920
  Val episodes     :          942
  Test episodes    :          986

  Derived stats:
    Train episodes / train user :       2.30
    Val episodes / val user     :       0.09
    Test episodes / test user   :       0.09
Chart saved: eda_xuetangx_05_episodes.png


---
## Pipeline Funnel — End-to-End Data Reduction

This section traces how the dataset shrinks at each processing stage.

In [8]:
print('=' * 68)
print('XUETANGX — FULL PIPELINE FUNNEL SUMMARY')
print('=' * 68)
print(f'  NB01 Raw events          : {m01["n_events"]:>12,}')
print(f'  NB01 Raw users           : {m01["n_users"]:>12,}')
print(f'  NB01 Raw courses         : {m01["n_courses"]:>12,}')
print()
print(f'  NB02 Sessionized events  : {m02["n_events"]:>12,}  (drop {(1-m02["n_events"]/m01["n_events"])*100:.1f}%)')
print(f'  NB02 Sessions created    : {m02["n_sessions"]:>12,}')
print(f'  NB02 Users with sessions : {m02["n_users"]:>12,}  (drop {(1-m02["n_users"]/m01["n_users"])*100:.1f}%)')
print()
print(f'  NB03 Vocabulary size     : {m03["n_items"]:>12,}')
print(f'  NB03 Total pairs         : {m03["n_pairs"]:>12,}')
print(f'  NB03 Users with pairs    : {m03["n_users"]:>12,}  (drop {(1-m03["n_users"]/m02["n_users"])*100:.1f}%)')
print()
print(f'  NB03b Sessions scored    : {m03b["n_sessions"]:>12,}')
print(f'  NB03b SRS mean / median  : {m03b["srs_mean"]:.4f} / {m03b["srs_p50"]:.4f}')
print()
print(f'  NB04 Users — Train       : {m04["n_users_train"]:>12,}  (70%)')
print(f'  NB04 Users — Val         : {m04["n_users_val"]:>12,}  (15%)')
print(f'  NB04 Users — Test        : {m04["n_users_test"]:>12,}  (15%)')
print()
print(f'  NB05 Train episodes      : {m05["n_episodes_train"]:>12,}  (K={m05["K"]}, Q={m05["Q"]})')
print(f'  NB05 Val episodes        : {m05["n_episodes_val"]:>12,}')
print(f'  NB05 Test episodes       : {m05["n_episodes_test"]:>12,}')
print('=' * 68)

# ── Funnel chart ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: User funnel
user_funnel = [m01['n_users'], m02['n_users'], m03['n_users'],
               m04['n_users_train'], m04['n_users_val'], m04['n_users_test']]
user_stages = ['NB01 Raw', 'NB02 With\nSessions', 'NB03 With\nPairs',
               'NB04 Train', 'NB04 Val', 'NB04 Test']
colors_u = ['#08519c', '#2171b5', '#4292c6', '#238b45', '#41ab5d', '#74c476']
bars = axes[0].bar(range(len(user_stages)), user_funnel, color=colors_u, edgecolor='white')
for bar, v in zip(bars, user_funnel):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1000,
                 f'{v:,}', ha='center', fontsize=8, rotation=30)
axes[0].set_xticks(range(len(user_stages)))
axes[0].set_xticklabels(user_stages, fontsize=9)
axes[0].set_ylabel('Users')
axes[0].set_title('User Count through Pipeline', fontsize=11)
axes[0].spines[['top', 'right']].set_visible(False)

# Right: Pairs / episodes per stage
pair_funnel = [m03['n_pairs'], m04['n_pairs_train'], m04['n_pairs_val'],
               m04['n_pairs_test'], m05['n_episodes_train'], m05['n_episodes_val'],
               m05['n_episodes_test']]
pair_stages = ['NB03 Pairs', 'Pairs Train', 'Pairs Val', 'Pairs Test',
               'Episodes Train', 'Episodes Val', 'Episodes Test']
colors_p = ['#54278f', '#756bb1', '#9e9ac8', '#cbc9e2',
            '#006d2c', '#31a354', '#74c476']
bars = axes[1].bar(range(len(pair_stages)), pair_funnel, color=colors_p, edgecolor='white')
for bar, v in zip(bars, pair_funnel):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1000,
                 f'{v:,}', ha='center', fontsize=8, rotation=30)
axes[1].set_xticks(range(len(pair_stages)))
axes[1].set_xticklabels(pair_stages, fontsize=9, rotation=20)
axes[1].set_ylabel('Count')
axes[1].set_title('Pairs & Episodes through Pipeline', fontsize=11)
axes[1].spines[['top', 'right']].set_visible(False)

plt.suptitle('XuetangX — Full Pipeline Data Reduction Funnel', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPO_ROOT / 'reports' / 'eda_xuetangx_funnel.png', dpi=150)
plt.show()
print('Chart saved: eda_xuetangx_funnel.png')

XUETANGX — FULL PIPELINE FUNNEL SUMMARY
  NB01 Raw events          :   28,002,537
  NB01 Raw users           :      182,755
  NB01 Raw courses         :        1,518

  NB02 Sessionized events  :   27,875,155  (drop 0.5%)
  NB02 Sessions created    :      906,341
  NB02 Users with sessions :      173,335  (drop 5.2%)

  NB03 Vocabulary size     :        1,517
  NB03 Total pairs         :      487,696
  NB03 Users with pairs    :       70,809  (drop 59.1%)

  NB03b Sessions scored    :      906,341
  NB03b SRS mean / median  : 0.3248 / 0.2456

  NB04 Users — Train       :       49,566  (70%)
  NB04 Users — Val         :       10,621  (15%)
  NB04 Users — Test        :       10,622  (15%)

  NB05 Train episodes      :      113,920  (K=5, Q=10)
  NB05 Val episodes        :          942
  NB05 Test episodes       :          986
Chart saved: eda_xuetangx_funnel.png


---
## Summary

| Stage | Output | Key Metric |
|-------|--------|------------|
| **NB01 Ingest** | 28,002,537 events | 182,755 users · 1,518 courses · 8 event types |
| **NB02 Sessionize** | 906,341 sessions | Gap threshold 1800 s · median gap 4 s |
| **NB03 Vocab+Pairs** | 487,696 pairs | 1,517-item vocabulary · 70,809 users |
| **NB03b SRS** | 906,341 sessions scored | SRS mean=0.32 · 62.8% low · 11.4% high |
| **NB04 User Split** | 70,809 users split | 70/15/15 · cold-start guarantee |
| **NB05 Episodes** | 115,848 total episodes | K=5 support · Q=10 query · 986 test episodes |

**Correlation between SRS and interaction depth** (from NB09):  
`r(SRS, n_events) = 0.503` — SRS correlates with intensity but is not reducible to event count.  
`r(SRS, duration) = 0.824` — session duration is the strongest single SRS driver.

The large imbalance toward low-SRS sessions (62.8%) motivates the SRS-Adaptive MAML design: noisy short sessions receive fewer adaptation steps, while high-quality long sessions drive gradient updates more strongly.